In [ ]:
import itertools
import os
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
from astropy import units as u
from astropy.visualization import quantity_support
from tqdm import tqdm
quantity_support()
import scipy as sp
# from scipy.stats._stats_py import Power_divergenceResult

from spectrum_component_analyser.readers.JWST.instruments import NIRISS, NIRSPECLower
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum
from spectrum_component_analyser.readers.JWST.file_reader import JWSTFileReader
from spectrum_component_analyser.readers.JWST.folder_reader import JWSTFolderReader
from spectrum_component_analyser.readers.JWST.target import *
from phoenix_grid_creator.spectral_grid import download_spectrum, get_phoenix_wavelengths, spectral_grid
from spectrum_component_analyser.spectrum import spectrum

target : JWSTTarget = K218

all_spectra : list[spectrum] = JWSTFolderReader.get_all_spectra(target=target, instrument=NIRISS)

print(all_spectra)

example_spectrum = all_spectra[0]

mask = np.isfinite(all_spectra[0].Fluxes)

s : spectrum
for s in all_spectra:
    s.plot(clear=False)
plt.show()

spec_grid : spectral_grid = spectral_grid.from_hdf5(grid_name="JWST_convolved_not_oversmoothed.hdf5")

In [ ]:
# draw light curve

indices = []
fluxes = []

s : spectrum
for index, s in enumerate(all_spectra):
    indices.append(index)
    fluxes.append(np.sum(s.Fluxes[mask].value))

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(indices, fluxes)
plt.show()

for s in all_spectra:
    s.normalise_flux()

In [ ]:
"""
find the standard deviation of each point
find an averaged spectrum
analyse that average spectrum using a (now correct) Pearson chi-squared test
"""

min_included_integration_index = 200
max_included_integration_index = 450

all_fluxes=[spec.Fluxes for spec in all_spectra[min_included_integration_index:max_included_integration_index]]

average_spectrum = spectrum(wavelengths=all_spectra[0].Wavelengths,
                                 fluxes=np.mean(all_fluxes, axis=0) * all_fluxes[0].unit,
                                 normalised_point=None,
                                 temperature = None,
                                 observational_resolution=None,
                                 observational_wavelengths=None,
                                 name="averaged " + target.name)

average_spectrum = average_spectrum[mask]

flux_std_deviation = np.std(all_fluxes, axis=0) * all_fluxes[0].unit

flux_std_deviation = flux_std_deviation[mask]

fig, ax = plt.subplots(figsize=(26,10))
average_spectrum.plot()
plt.errorbar(average_spectrum.Wavelengths, average_spectrum.Fluxes, yerr=flux_std_deviation, ecolor='red', capsize=2)
plt.scatter(average_spectrum.Wavelengths, flux_std_deviation, s=1)
plt.show()


In [ ]:
"""
https://www.aanda.org/articles/aa/pdf/2016/03/aa22261-13.pdf talks about a dynamical mask, could maybe introduce that?

e.g. using

total_fluxes : np.ndarray = np.zeros((len(spec_grid.Wavelengths))) * u.Jy

all_phoenix_params = list(itertools.product(spec_grid.T_effs, spec_grid.FeHs, spec_grid.Log_gs))

for t, f, l in all_phoenix_params:
    total_fluxes += spec_grid.LookupTable[t, f, l]

average_flux = total_fluxes / len(all_phoenix_params)

average_flux = average_flux[mask]
"""

number_of_components : int = 1
number_of_parameters : int = 4

def compute_error(params):
    observed = average_spectrum.Fluxes.copy()
    simulated : np.ndarray = []
    
    for i in range(number_of_components):
        weight = params[0 + i * number_of_parameters]
        t = params[1 + i * number_of_parameters]
        f = params[2 + i * number_of_parameters]
        l = params[3 + i * number_of_parameters]

        t *= u.K
        f *= u.dex
        l *= u.dex

        spec : phoenix_spectrum = get_interpolated_phoenix_spectrum(t, f, l, star_name=target.name, spec_grid=spec_grid)

        spec = spec[mask]

        if len(simulated) == 0:
            simulated = weight * spec.Fluxes
        else:
            simulated += weight * spec.Fluxes

    e = np.sum(((observed - simulated)**2).value)

    return e

parameter_bounds = [
        (0, 2),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
        # (target.feh.value * .9, target.feh.value * 1.1),
        # (target.log_g.value * .9, target.log_g.value * 1.1)
]

initial_guess = [
    1,
    target.t_eff,
    target.feh,
    target.log_g
]

r = sp.optimize.differential_evolution(
    func=compute_error,
    # x0 = initial_guess * number_of_components,
    bounds=parameter_bounds * number_of_components,
    callback=print
)

print(r)

# with ProcessPoolExecutor() as executor:
#     results = list(tqdm(executor.map(compute_error, params), total=len(params)))

# chi2_grid = np.array(results).reshape((len(ts), len(fehs), len(loggs)))
# chi2_grid *= u.Jy

In [ ]:
%matplotlib inline

total_fluxes = []

for i in range(number_of_components):
    w_title : str = f"weight {i}"
    t_title : str = f"teff {i}"
    f_title : str = f"feh {i}"
    l_title : str = f"log_g {i}"

    w = r.x[0 + number_of_parameters*i]
    t = r.x[1 + number_of_parameters*i] * u.K
    f = r.x[2  + number_of_parameters*i] * u.dex
    l = r.x[3 + number_of_parameters*i] * u.dex

    print(f"{w_title:<20} : {w}")
    print(f"{t_title:<20} : {t}")
    print(f"{f_title:<20} : {f}")
    print(f"{l_title:<20} : {l}")
    print("\n")

    
    componentised_spectrum = get_interpolated_phoenix_spectrum(t, f, l, star_name=target.name + f" component {i}", spec_grid=spec_grid)

    componentised_spectrum = componentised_spectrum[mask]

    componentised_spectrum.Fluxes *= w

    componentised_spectrum.plot(clear=False)

    if len(total_fluxes) == 0:
        total_fluxes = componentised_spectrum.Fluxes
    else:
        total_fluxes += componentised_spectrum.Fluxes

plt.legend()
plt.show()

# plot components neatly
# make subplot with: literature belief parameters & residuals, our 1 component solution & residuals, our 2 / 3 etc component solution

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(30, 16), sharex=True)

# get literature belief
target_interpolated : phoenix_spectrum = get_interpolated_phoenix_spectrum(
        T_eff=target.t_eff,
        FeH=target.feh,
        Log_g=target.log_g,
        star_name=target.name + " literature belief",
        spec_grid=spec_grid
        )
target_interpolated = target_interpolated[mask]
ax1.plot(average_spectrum.Wavelengths, average_spectrum.Fluxes, label="jwst observation", color="black")
ax1.plot(target_interpolated.Wavelengths, target_interpolated.Fluxes, label="literature", color="orange")
ax1.plot(spec_grid.Wavelengths[mask], total_fluxes, label="my result (using components)", color="blue")

literature_residuals = (target_interpolated.Fluxes - average_spectrum.Fluxes) / average_spectrum.Fluxes
my_residuals = (total_fluxes - average_spectrum.Fluxes) / average_spectrum.Fluxes

ax2.plot(average_spectrum.Wavelengths, literature_residuals, label="literature", color="orange")
ax2.plot(average_spectrum.Wavelengths, my_residuals, label="my result (using components)", color="blue")
ax2.set_ylabel(r"Residual = $\frac{\mathrm{Fitted\ Flux}-\mathrm{Observed\ Flux}}{\mathrm{Observed\ Flux}}$")

ax1.legend()
ax2.legend()

plt.show()

print(np.sum(np.abs(literature_residuals)))
print(np.sum(np.abs(my_residuals)))


In [ ]:
# chi2_grid = np.zeros((len(ts), len(fehs), len(loggs)))

# Label the best fit point
min_idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
optimal_t = ts[min_idx[0]]
optimal_feh = fehs[min_idx[1]]
optimal_logg = loggs[min_idx[2]]
min_chi2 = chi2_grid[min_idx]

%matplotlib inline
k_slice = min_idx[2]
selected_logg = optimal_logg

# Slice the 3D grid to get a 2D (Teff, FeH) array
plot_data = chi2_grid[:, :, k_slice]


plt.figure(figsize=(10, 7))

# Create the plot
# Shading='gouraud' smooths the grid; 'auto' keeps it pixelated
pcm = plt.pcolormesh(fehs, ts, plot_data, cmap='viridis_r', shading='auto')

# Add a colorbar to show the chi-squared magnitude
cbar = plt.colorbar(pcm)
cbar.set_label(r'$\chi^2$ (Residual Sum of Squares)', rotation=270, labelpad=15)


print(optimal_t, optimal_feh, optimal_logg)
print(min_chi2)

plt.scatter(optimal_feh, optimal_t, color='red', marker='x', s=100, label='Best Fit')

plt.xlabel('Metallicity [Fe/H]')
plt.ylabel('Effective Temperature $T_{eff}$ [K]')
plt.title(f'Chi-Squared Surface for log(g) = {loggs[0]}')
plt.legend()
plt.show()

average_spectrum.plot()
i = get_interpolated_phoenix_spectrum(optimal_t, optimal_feh, optimal_logg, star_name="interpolated", spec_grid=spec_grid)
# i = get_interpolated_phoenix_spectrum(3300 * u.K, .0 * u.dex, 4.85 * u.dex, star_name="test", spec_grid=spec_grid)
# i.Fluxes /= normalisation
i.plot(clear=False)
plt.legend()
plt.show()

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import numpy as np

# 1. Setup the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))
plt.subplots_adjust(bottom=0.25) # Make room for the slider

# 2. Initial plot (default to the first logg)
k_init = 0
# Use np.log as you did in your snippet
pcm = ax.pcolormesh(fehs, ts, chi2_grid[:, :, k_init], 
                    cmap='viridis_r', shading='auto')

# 3. Add formatting and the 'Best Fit' marker
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r'$\ln(\chi^2)$', rotation=270, labelpad=15)

# Calculate global best fit once
min_idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
best_i, best_j, best_k = min_idx

# We only show the marker if it's on the current slice (optional logic)
fit_marker = ax.scatter(fehs[best_j], ts[best_i], color='red', 
                        marker='x', s=100, label='Global Best Fit')

ax.set_xlabel('Metallicity [Fe/H]')
ax.set_ylabel('Effective Temperature $T_{eff}$ [K]')
ax_title = ax.set_title(f'Chi-Squared Surface: log(g) = {loggs[k_init]:.2f}')
ax.legend()

# 4. Create the Slider axes [left, bottom, width, height]
ax_slider = plt.axes([0.2, 0.1, 0.6, 0.03])
slider = Slider(
    ax=ax_slider,
    label='log(g) ',
    valmin=0,
    valmax=len(loggs) - 1,
    valinit=k_init,
    valfmt='%d' # Display as integer index
)

# 5. The Update Function
def update(val):
    k = int(slider.val)
    # Update the data in the pcolormesh
    new_data = np.log(chi2_grid[:, :, k])
    pcm.set_array(new_data.ravel())
    
    # Update title and visibility of the best-fit marker
    ax_title.set_text(f'Chi-Squared Surface: log(g) = {loggs[k]:.2f}')
    
    # Optional: Highlight best fit only if on the correct slice
    fit_marker.set_visible(k == best_k)
    
    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()